In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
from scipy.stats import gaussian_kde

figure_dir = "figures"

In [ ]:
adata = sc.read_h5ad("data/xenium/processed/08.1-kidney_tcr_clonal_clusters.h5ad")

In [ ]:
tmp = adata.copy()

In [ ]:
len([x for x in tmp.var_names if x.startswith("TRAV")])

In [ ]:
# Rename specific gene names
merged_genes = [
    "TRBV10*",
    "TRBV11*",
    "TRBV12*",
    "TRBV18_19*",
    "TRBV4*",
    "TRBV5-6*",
    "TRBV6*",
    "TRBV7-2_3*",
    "TRAV19_21*",
    "TRAV12*",
    "TRAV8-4*",
    "TRAV9*",
]
rename_dict = {mg[:-1]: mg for mg in merged_genes}
tmp.var_names = tmp.var_names.to_series().replace(rename_dict)

In [ ]:
cond_dic = {
    "Control": [
        "XETG00088__0029040__Region_2__20240719__095641",
        "XETG00088__0029040__Region_3__20240719__095641",
        "XETG00088__0029041__Region_1__20240719__095642",
    ],
    "ANCA": [
        "XETG00088__0029040__Region_4__20240719__095642",
        "XETG00088__0029040__Region_5__20240719__095642",
        "XETG00088__0029040__Region_6__20240719__095642",
        "XETG00088__0029040__Region_7__20240719__095642",
        "XETG00088__0029041__Region_3__20240719__095642",
        "XETG00088__0029041__Region_4__20240719__095642",
        "XETG00088__0029041__Region_5__20240719__095642",
        "XETG00088__0029041__Region_6__20240719__095642",
        "XETG00088__0029041__Region_7__20240719__095642",
        "XETG00088__0029041__Region_8__20240719__095642",
    ],
    "BK-Virus": ["XETG00088__0029041__Region_2__20240719__095642"],
}
# inverted_dict = {sample_id: condition for condition, sample_ids in cond_dic.items() for sample_id in sample_ids}

# # Assuming tmp.obs["sample"] contains the sample IDs
# tmp.obs["condition"] = tmp.obs["sample"].replace(inverted_dict)

In [ ]:
# only concentrate on the vgenes
av = [x for x in tmp.var_names if x.startswith("TRAV")]
bv = [x for x in tmp.var_names if x.startswith("TRBV")]
gv = [x for x in tmp.var_names if x.startswith("TRGV")]
dv = [x for x in tmp.var_names if x.startswith("TRDV")]

In [ ]:
def get_dominant_v(tmp, vgenes):
    array = tmp[:, vgenes].X.toarray()
    max_indices = np.argmax(array, axis=1)
    _max_values = array[np.arange(array.shape[0]), max_indices]
    # pdb.set_trace()
    sorted_rows = np.sort(array, axis=1)[:, -2:]
    ties = sorted_rows[:, 0] == sorted_rows[:, 1]
    result = np.where(ties, -1, max_indices)
    # pdb.set_trace()
    return [vgenes[i] if i != -1 else "None" for i in result]


tmp.obs["AV"] = get_dominant_v(tmp, av)
tmp.obs["BV"] = get_dominant_v(tmp, bv)

# Preprocess  tmp add columns = [IncompleteTCR, AV, BV, ]

In [ ]:
realt = ["CD4+", "CD8+", "MAIT", "NK/NKT", "TFH", "Tregs", "gdT"]

In [ ]:
tidx = tmp[~tmp.obs.tcell_subtype.isin(realt)].obs.index
tmp.obs.loc[tidx, "AV"] = "None"
tmp.obs.loc[tidx, "BV"] = "None"

In [ ]:
tmp.obs.loc[:, "clonesize"] = 0

In [ ]:
tmp.obs["IncompleteTCR"] = tmp.obs[["AV", "BV"]].apply(lambda x: "None" in x.values, 1)

In [ ]:
tmp.obs.loc[~tmp.obs["IncompleteTCR"], ["AV", "BV"]].value_counts()

In [ ]:
tmp.obs.loc[:, "realt"] = tmp.obs.tcell_subtype.isin(realt)

# Zoomed in Areas

In [ ]:
sample = "XETG00088__0029041__Region_8__20240719__095642"

In [ ]:
asample = tmp[tmp.obs["sample"] == sample].copy()

# Categorize expanded vs Singlets

In [ ]:
mapbol = (asample.obs[["AV", "BV"]] != "None").all(1)
vc = asample[mapbol].obs[["AV", "BV"]].value_counts()
expandedclones = vc[vc > 1].index
singleclones = vc[vc == 1].index

In [ ]:
srealt = asample[asample.obs.realt].copy()


def get_idx_clone(clones):
    idx = []
    for clone in clones:
        gena = clone[0]
        genb = clone[1]
        amap = srealt.obs["AV"] == gena
        bmap = srealt.obs["BV"] == genb
        clonmap = np.logical_and(amap, bmap)
        idx += list(srealt[clonmap].obs.index)
    return idx

In [ ]:
expidx = get_idx_clone(expandedclones)
singidx = get_idx_clone(singleclones)

In [ ]:
asample.obs.loc[expidx, "expansion"] = "expanded"
asample.obs.loc[singidx, "expansion"] = "singles"

In [ ]:
# --- Color Setup ---
custom_palette = {"CD4+": "red", "CD8+": "blue"}

# --- Filter: only relevant groups ---
# asample = asample[asample.obs["expansion"].isin(["expanded", "singles"])].copy()
# asample.obs["expansion"] = asample.obs["expansion"].astype("category")

# Coordinates
coords = asample.obsm["spatial"]
x = coords[:, 0]
y = coords[:, 1]
x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

# --- KDE height/width settings ---
kde_height = (y_max - y_min) * 0.15
kde_width = (x_max - x_min) * 0.15

# --- Grid for KDEs ---
x_grid = np.linspace(x_min, x_max, 500)
y_grid = np.linspace(y_min, y_max, 500)

# --- Group-wise KDEs ---
kde_lines_x = {}
kde_lines_y = {}

for group, color in custom_palette.items():
    mask = asample.obs["tcell_subtype"] == group
    x_group = x[mask]
    y_group = y[mask]

    if len(x_group) > 1:  # Avoid error in KDE
        kde_x = gaussian_kde(x_group, bw_method=0.02)
        kde_y = gaussian_kde(y_group, bw_method=0.02)

        density_x = kde_x(x_grid)
        density_y = kde_y(y_grid)

        kde_y_scaled = y_max + (density_x / density_x.max()) * kde_height
        kde_x_scaled = x_max + (density_y / density_y.max()) * kde_width

        kde_lines_x[group] = (x_grid, kde_y_scaled, color)
        kde_lines_y[group] = (kde_x_scaled, y_grid, color)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 8))
groups_to_plot = list(custom_palette.keys())
asample_tmp = asample.copy()
asample_tmp.obs["tcell_subtype"] = asample_tmp.obs["tcell_subtype"].cat.add_categories(
    ["other"]
)
asample_tmp.obs["tcell_subtype"] = asample_tmp.obs["tcell_subtype"].where(
    asample_tmp.obs["tcell_subtype"].isin(groups_to_plot), other="other"
)
asample_tmp.obs["tcell_subtype"] = asample_tmp.obs[
    "tcell_subtype"
].cat.remove_unused_categories()

palette = {**custom_palette, "other": "#d3d3d3"}
# Spatial plot
sc.pl.spatial(
    asample_tmp,
    ax=ax,
    color="tcell_subtype",
    groups=groups_to_plot,
    colorbar_loc=None,
    palette=palette,
    vmax=1,
    spot_size=10,
    show=False,
    title="T cell spatial distribution: expanded vs singles",
    linewidth=0.5,
)

# Overlay KDEs
for group in kde_lines_x:
    xg, yg, col = kde_lines_x[group]
    ax.plot(xg, yg, color=col, lw=1, label=group)
    ax.fill_between(xg, y_max, yg, color=col, alpha=0.3)

for group in kde_lines_y:
    xg, yg, col = kde_lines_y[group]
    ax.plot(xg, yg, color=col, lw=1)
    ax.fill_betweenx(yg, x_max, xg, color=col, alpha=0.3)

# Adjust limits
ax.set_xlim(x_min, x_max + kde_width * 1.1)
ax.set_ylim(y_min, y_max + kde_height * 1.1)
# Place legend outside the plot (right side)
ax.legend(
    title="Expanded status",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),  # outside right, vertically centered
    borderaxespad=0,
    frameon=False,
)

plt.tight_layout(pad=0, rect=[0, 0, 0.85, 1])  # leave space for legend
# plt.legend(title="Expanded status", loc="upper right")
# plt.tight_layout(pad=0)
plt.savefig(f"{figure_dir}/03spatial_CD4CD8.png", dpi=300)